In [ ]:
import re
import torch
import pandas as pd
from torch.utils.data import Dataset, DataLoader  
from transformers import BertTokenizerFast, BertModel 
import torch.nn as nn  
from sklearn.metrics import accuracy_score
from torch.optim import AdamW
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, confusion_matrix
from tqdm import tqdm
import time


train_df = pd.read_excel('/kaggle/input/only-urdu-mmhs11k/MMHS11K_train.xlsx')
test_df = pd.read_excel('/kaggle/input/only-urdu-mmhs11k/MMHS11K_test.xlsx')

# Map 'hate' to 1 and 'no_hate' to 0
train_df['Label'] = train_df['Label'].map({'No_Hate': 0, 'Hate': 1})
test_df['Label'] = test_df['Label'].map({'No_Hate': 0, 'Hate': 1})

def clean_urdu_text(text):
  # Remove URLs and user mentions
  text = re.sub(r'https?://\S+|www\.\S+', '', str(text))
  text = re.sub(r'@\w+', '', text)
  return text.strip()

test_df['Text'] = test_df['Text'].apply(clean_urdu_text)
test_df['Word_Count'] = test_df['Text'].apply(lambda x: len(x.split()))

X_train = train_df['Text'].values
X_test = test_df['Text'].values

y_train = train_df['Label'].values
y_test = test_df['Label'].values

# Define the TextDataset class for tokenization and batching
class TextDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length):
        self.texts = texts.reset_index(drop=True)  # Ensure consistent indexing of texts
        self.labels = labels.reset_index(drop=True)  # Ensure consistent indexing of labels
        self.tokenizer = tokenizer  # Save the tokenizer instance
        self.max_length = max_length  # Save max sequence length for padding/truncation
    def __len__(self):
        return len(self.texts)  # Return total number of samples
    def __getitem__(self, idx):
        text = str(self.texts[idx])  # Get text at index idx and ensure it's a string
        label = self.labels[idx]  # Get corresponding label

        # Tokenize the text using BERT tokenizer
        encoding = self.tokenizer.encode_plus(
            text,  # Input text
            add_special_tokens=True,  # Add special tokens like [CLS] and [SEP]
            max_length=self.max_length,  # Pad/truncate to fixed length
            return_token_type_ids=False,  # Not needed for single-sentence classification
            padding='max_length',  # Apply padding
            truncation=True,  # Truncate if input is too long
            return_attention_mask=True,  # Generate attention mask
            return_tensors='pt',  # Return as PyTorch tensors
        )

        # Return a dictionary containing input_ids, attention_mask, and label tensor
        return {
            'input_ids': encoding['input_ids'].flatten(),  # Convert to 1D tensor
            'attention_mask': encoding['attention_mask'].flatten(),  # 1D attention mask
            'labels': torch.tensor(label, dtype=torch.long)  # Label as tensor
        }

# Training hyperparameters
num_epoch = 20  # Total number of training epochs
sq_len = 128  # Max sequence length for BERT input
batch_size = 32  # Number of samples in each training batch
lr_rate = 2e-5  # Learning rate for optimizer
epsilon = 1e-8  # Small value to prevent divide-by-zero in Adam optimizer
# hidden_dropout = 0.05  # Dropout to prevent overfitting
hidden_dropout = 0.3  # Dropout to prevent overfitting
warmup_ratio = 0.06  # Used if you apply learning rate warm-up (not applied yet)
weight_decay = 0.01  # Weight decay for L2 regularization

# Load the multilingual BERT tokenizer
tokenizer = BertTokenizerFast.from_pretrained("bert-base-multilingual-cased")  # Load pre-trained tokenizer

from torch.utils.data import DataLoader

# Prepare train/test TextDataset objects
train_dataset = TextDataset(
    texts=train_df['Text'],
    labels=train_df['Label'],  # Or whatever your label column is
    tokenizer=tokenizer,
    max_length=sq_len
)

test_dataset = TextDataset(
    texts=test_df['Text'],
    labels=test_df['Label'],  # Use test labels if available
    tokenizer=tokenizer,
    max_length=sq_len
)

# Wrap into DataLoader for batching
train_dataloader = DataLoader(train_dataset, batch_size, shuffle=True)
test_dataloader = DataLoader(test_dataset, batch_size)


# Define a custom model class EnhancedBERT that extends PyTorch's nn.Module
class EnhancedBERT(nn.Module):
    def __init__(self):
        super(EnhancedBERT, self).__init__()  # Call the base class constructor

        # Load pre-trained BERT (multilingual, cased)
        self.bert = BertModel.from_pretrained('bert-base-multilingual-cased')  

        # Dropout layer to prevent overfitting (value comes from earlier: hidden_dropout = 0.05)
        self.dropout = nn.Dropout(hidden_dropout)

        # Fully connected layer for classification
        # BERT's hidden size is 768, and we have 2 output classes (binary classification)
        self.fc = nn.Linear(self.bert.config.hidden_size, 2)

    def forward(self, input_ids, attention_mask):
        # Pass the input_ids and attention_mask to BERT
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)

        # Extract the pooled output (corresponds to the [CLS] token representation)
        pooled_output = outputs.pooler_output

        # Apply dropout to the [CLS] token embedding
        pooled_output = self.dropout(pooled_output)

        # Pass the result through the final classification layer to get logits
        logits = self.fc(pooled_output)

        # Return the logits (raw scores for each class)
        return logits


# Use GPU if available, otherwise fallback to CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Create an instance of the EnhancedBERT model
model = EnhancedBERT()

# Use DataParallel to utilize multiple GPUs
if torch.cuda.device_count() > 1:
    print(f"✅ Using {torch.cuda.device_count()} GPUs")
    model = nn.DataParallel(model)

# Move model to GPU/CPU
model = model.to(device)

# Define optimizer
optimizer = AdamW(model.parameters(), lr=lr_rate, eps=epsilon, weight_decay=weight_decay)

# Define loss function
loss_fn = nn.CrossEntropyLoss()


# Initialize metric storage
epoch_list = []
training_loss_list = []
training_time_list = []
test_accuracy_list = []
test_f1_list = []
test_precision_list = []
test_recall_list = []
test_tp_list = []
test_tn_list = []
test_fp_list = []
test_fn_list = []

def train_model(model, train_dataloader, test_dataloader, optimizer, loss_fn, epochs=5):
    best_test_loss = float('inf')  # Track the best test loss
    best_test_accuracy = 0.0
    
    for epoch in range(epochs):
        start_train = time.time()
        model.train()
        total_train_loss = 0

        train_loop = tqdm(train_dataloader, desc=f"Epoch {epoch+1}/{epochs} - Training", leave=False)
        for batch in train_loop:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            optimizer.zero_grad()
            logits = model(input_ids, attention_mask)
            loss = loss_fn(logits, labels)
            total_train_loss += loss.item()
            loss.backward()
            optimizer.step()

            train_loop.set_postfix(loss=loss.item())

        end_train = time.time()

        # Evaluate on test data
        model.eval()
        test_preds, test_labels = [], []
        total_test_loss = 0
        
        save_model_acc = False
        save_model_loss = False

        with torch.no_grad():
            for batch in test_dataloader:
                input_ids = batch['input_ids'].to(device)
                attention_mask = batch['attention_mask'].to(device)
                labels = batch['labels'].to(device)

                logits = model(input_ids, attention_mask)
                loss = loss_fn(logits, labels)
                total_test_loss += loss.item()

                preds = torch.argmax(logits, dim=1)
                test_preds.extend(preds.cpu().numpy())
                test_labels.extend(labels.cpu().numpy())

        avg_test_loss = total_test_loss / len(test_dataloader)

        test_accuracy = accuracy_score(test_labels, test_preds)
        test_f1 = f1_score(test_labels, test_preds, average='weighted')
        test_precision = precision_score(test_labels, test_preds, average='weighted', zero_division=0)
        test_recall = recall_score(test_labels, test_preds, average='weighted', zero_division=0)
        tn, fp, fn, tp = confusion_matrix(test_labels, test_preds, labels=[0, 1]).ravel()

        # Save best model based on test loss
        if avg_test_loss < best_test_loss:
            best_test_loss = avg_test_loss
            torch.save(model.state_dict(), f'best_model_Enhanced_BERT_loss_2e-5.pt')
            print(f"\n✅ Best model saved at epoch {epoch+1} (test_loss: {avg_test_loss:.5f})")
            save_model_loss = True

        if test_accuracy > best_test_accuracy:
            best_test_accuracy = test_accuracy
            torch.save(model.state_dict(), f'best_model_Enhanced_BERT_accuracy_2e-5.pt')
            print(f"\n✅ Model saved at epoch {epoch+1} due to improved accuracy: {test_accuracy:.5f}")
            save_model_acc = True
            
        if not save_model_acc and not save_model_loss:
            print(f"⚠️  No improvement at epoch {epoch+1} (Acc: {test_accuracy:.5f}, Loss: {avg_test_loss:.5f})")


        # Logging
        epoch_list.append(epoch + 1)
        training_loss_list.append(total_train_loss / len(train_dataloader))
        training_time_list.append(time.strftime("%H:%M:%S", time.gmtime(end_train - start_train)))
        test_accuracy_list.append(test_accuracy)
        test_f1_list.append(test_f1)
        test_precision_list.append(test_precision)
        test_recall_list.append(test_recall)
        test_tp_list.append(tp)
        test_tn_list.append(tn)
        test_fp_list.append(fp)
        test_fn_list.append(fn)

        # Epoch Summary
        print(f"\n📢 Epoch {epoch + 1}/{epochs}")
        print(f"Train Loss: {training_loss_list[-1]:.5f} | Test Loss: {avg_test_loss:.5f}")
        print(f"Test Accuracy: {test_accuracy:.5f} | Test F1: {test_f1:.5f}")
        print(f"TP: {tp}, TN: {tn}, FP: {fp}, FN: {fn}")

# Run training
train_model(model, train_dataloader, test_dataloader, optimizer, loss_fn, num_epoch)

# Convert results to DataFrames
train_df = pd.DataFrame({
    "Epoch": epoch_list,
    "Training Loss": training_loss_list,
    "Training Time": training_time_list
})

test_df = pd.DataFrame({
    "Epoch": epoch_list,
    "Test TP": test_tp_list,
    "Test TN": test_tn_list,
    "Test FP": test_fp_list,
    "Test FN": test_fn_list,
    "Test Accuracy": test_accuracy_list,
    "Test F1": test_f1_list,
    "Test Precision": test_precision_list,
    "Test Recall": test_recall_list
})

# Save to CSV
train_df.to_csv("training_metrics.csv", index=False)
test_df.to_csv("test_metrics.csv", index=False)

# Display summaries
print("\n📊 Training Summary:")
print(train_df)

print("\n🧪 Testing Summary:")
print(test_df)
